# Qwen3 distillation — corpus teacher cố định, Kaggle pilot

Upload notebook này lên Kaggle, bật **GPU + Internet**. Không cần clone repo hoặc GitHub token.
Source train đã được đóng gói. Notebook tải duy nhất file teacher Qwen v2; không tải/chạy teacher.

Mặc định: Qwen3-0.6B, 5.000 mẫu train tối đa sau lọc, 200 bước mỗi nhánh, LoRA rank 16,
sequence tối đa 2.048, micro-batch 1 × accumulation 8. Chạy **uniform** và **progress** tuần tự.
Đây là pilot để kiểm tra phương pháp, không phải tái lập paper hay kết quả benchmark cuối.
Đổi sang Qwen3-1.7B ở cell cấu hình để thử model thứ hai.

Pipeline: tải → audit → xác nhận mapping → prepare/split → kiểm tra runtime → train → so sánh log → export.
**Không Run All ngay lần đầu**: đọc dữ liệu tại bước audit và xác nhận prompt/response trước khi prepare.
Không có kết quả train Kaggle được xác nhận trước; thời gian/VRAM phụ thuộc GPU và token thực tế.


In [ ]:
from pathlib import Path
import json, os, sys, subprocess, hashlib, shutil
BASE = Path('/kaggle/working/distill-pilot' if Path('/kaggle').exists()
            else '/content/distill-pilot' if Path('/content').exists() else './distill-pilot').resolve()
BASE.mkdir(parents=True, exist_ok=True)
MODEL = 'Qwen/Qwen3-0.6B'  # Lần sau: 'Qwen/Qwen3-1.7B'
MODEL_REVISION = 'main'  # prepare sẽ resolve SHA cụ thể và lưu vào manifest
DATASET = 'zhang-yice/sentiment-distillation-v2'
DATASET_REVISION = '0ee48fc022bb45ad9f2c66c43fc1c73db6c3f4fb'
TEACHER_FILE = 'Qwen3-32B_50k.jsonl'
LOCAL_TEACHER_FILE = None  # Hoặc '/kaggle/input/.../Qwen3-32B_50k.jsonl'
INSTALL_DEPENDENCIES = True  # Giữ CUDA torch của Kaggle; có dry-run bảo vệ torch
FOUR_BIT = False  # LoRA thường trước; True thêm bitsandbytes và bật QLoRA
RUN_TAG = 'pilot-v1'  # Đổi tên nếu thay cấu hình; không ghi đè thí nghiệm cũ
MAX_TRAIN = 5000  # 0 = toàn bộ train hợp lệ sau grouped holdout
MAX_LENGTH = 2048  # Mẫu dài hơn bị loại nguyên mẫu, không cắt đáp án
STEPS = 200
MODES = ['uniform', 'progress']  # Ablation: thêm 'balanced', 'loss'
SEED = 42
GRAD_ACCUM = 8
LEARNING_RATE = 1e-4
PROBE_EVERY = 25
SAVE_EVERY = 100
RESUME = {}  # Ví dụ {'progress': '/kaggle/working/.../progress/checkpoint-100'}
EXPORT_MODE = None  # Đặt 'uniform' hoặc 'progress' sau khi train xong
MODEL_TAG = MODEL.split('/')[-1]
WORK = BASE / RUN_TAG / MODEL_TAG
WORK.mkdir(parents=True, exist_ok=True)
print('Work:', WORK)


In [ ]:
BUNDLED = {'training/distill_core.py': '"""Dependency-free data checks, splits and adaptive group weights."""\nimport hashlib\nimport json\nimport math\nimport random\nfrom collections import Counter, defaultdict\nfrom pathlib import Path\n\n\ndef digest(value):\n    return hashlib.sha256(json.dumps(value, ensure_ascii=False, sort_keys=True).encode()).hexdigest()\n\n\ndef file_hash(path):\n    h = hashlib.sha256()\n    with open(path, \'rb\') as stream:\n        for chunk in iter(lambda: stream.read(1024 * 1024), b\'\'):\n            h.update(chunk)\n    return h.hexdigest()\n\n\ndef read_records(path):\n    with open(path, encoding=\'utf-8-sig\') as stream:\n        for line, text in enumerate(stream, 1):\n            if not text.strip():\n                continue\n            try:\n                obj = json.loads(text)\n                if not isinstance(obj, dict):\n                    raise ValueError(\'record is not an object\')\n                yield line, obj\n            except (ValueError, TypeError) as exc:\n                raise ValueError(f\'Invalid JSON record at line {line}: {exc}\') from exc\n\n\ndef mapped_record(obj, prompt_fields, response_field, group_field, source_field):\n    """Explicit field order; preserve each field\'s contents, including whitespace."""\n    parts = []\n    for field in prompt_fields:\n        value = obj.get(field)\n        if value is None:\n            raise ValueError(f\'missing prompt field: {field}\')\n        if not isinstance(value, str):\n            raise ValueError(f\'prompt field must be a string: {field}\')\n        if value.strip():\n            parts.append(value)\n    response = obj.get(response_field)\n    if not parts or not isinstance(response, str) or not response.strip():\n        raise ValueError(\'empty prompt or response\')\n    group = obj.get(group_field)\n    if group is None or not str(group).strip():\n        raise ValueError(f\'missing group field: {group_field}\')\n    source = obj.get(source_field)\n    if not isinstance(source, str) or not source.strip():\n        raise ValueError(f\'missing source/input field for group split: {source_field}\')\n    # Normalize only the split key; never alter the training target.\n    source_key = \' \'.join(source.split()).casefold()\n    return {\'prompt\': \'\\n\\n\'.join(parts), \'response\': response,\n            \'group\': str(group), \'source_key\': digest(source_key)}\n\n\ndef split_records(records, seed, dev_fraction=0.1, max_train=0, probe_per_group=4,\n                  holdout_per_group=4, min_group_train=8, max_groups=32):\n    if not 0 < dev_fraction < 1:\n        raise ValueError(\'dev_fraction must be between 0 and 1\')\n    if min(probe_per_group, holdout_per_group, min_group_train, max_groups) < 1 or max_train < 0:\n        raise ValueError(\'Invalid split limits\')\n    # Assign every input group before selecting a pilot subset. No input crosses splits.\n    keys = sorted({r[\'source_key\'] for r in records})\n    random.Random(seed).shuffle(keys)\n    ndev = max(2, round(len(keys) * dev_fraction))\n    if ndev >= len(keys):\n        raise ValueError(\'Not enough independent inputs to split\')\n    dev_keys = set(keys[:ndev])\n    probe_keys = set(keys[:ndev // 2])\n    train = [r for r in records if r[\'source_key\'] not in dev_keys]\n    random.Random(seed).shuffle(train)\n    if max_train:\n        train = train[:max_train]\n    counts = Counter(r[\'group\'] for r in train)\n    # Group collapsing uses TRAIN only. Reserve one slot for rare groups.\n    common = [g for g, n in sorted(counts.items(), key=lambda x: (-x[1], x[0]))\n              if n >= min_group_train][:max_groups - 1]\n    mapping = {g: g if g in common else \'__rare__\' for g in counts}\n    groups = sorted(set(mapping.values()))\n    for r in train:\n        r[\'train_group\'] = mapping[r[\'group\']]\n    probe, holdout = [], []\n    for allowed, dest, limit in ((probe_keys, probe, probe_per_group),\n                                  (dev_keys - probe_keys, holdout, holdout_per_group)):\n        buckets = defaultdict(list)\n        for r in records:\n            if r[\'source_key\'] in allowed and r[\'group\'] in mapping:\n                r[\'train_group\'] = mapping[r[\'group\']]\n                buckets[r[\'train_group\']].append(r)\n        for g in groups:\n            items = sorted(buckets[g], key=lambda r: r[\'id\'])\n            random.Random(f\'{seed}:{g}\').shuffle(items)\n            dest.extend(items[:limit])\n    if not train or not probe or not holdout:\n        raise ValueError(\'Empty train/probe/holdout; increase pilot size or dev fraction\')\n    return train, probe, holdout, mapping\n\n\nclass GroupWeights:\n    """Progress normalized by initial per-response CE; bounded mean-one weights."""\n    def __init__(self, groups, counts, mode=\'progress\', mix=0.5, temperature=0.05,\n                 ema=0.5, floor=0.5, ceiling=2.0):\n        if mode not in (\'uniform\', \'balanced\', \'loss\', \'progress\'):\n            raise ValueError(\'Unknown weighting mode\')\n        if not 0 <= mix <= 1 or temperature <= 0 or not 0 <= ema < 1:\n            raise ValueError(\'Invalid mix/temperature/EMA\')\n        if not 0 < floor <= 1 <= ceiling:\n            raise ValueError(\'Require 0 < floor <= 1 <= ceiling\')\n        self.groups = list(groups)\n        self.counts = dict(counts)\n        self.mode, self.mix, self.temperature, self.ema = mode, mix, temperature, ema\n        self.floor, self.ceiling = floor, ceiling\n        self.initial, self.previous, self.smoothed = {}, {}, {}\n        self.weights = {g: 1.0 for g in groups}\n        if mode == \'balanced\':\n            self.weights = self._bounded({g: 1 / counts[g] for g in groups})\n\n    def _bounded(self, raw):\n        # Solve E_train[clip(scale * raw, floor, ceiling)] = 1.\n        total = sum(self.counts.values())\n        lo, hi = 0., 1.\n        def values(scale):\n            return {g: min(self.ceiling, max(self.floor, scale * raw[g])) for g in self.groups}\n        def mean(w):\n            return sum(w[g] * self.counts[g] for g in self.groups) / total\n        while mean(values(hi)) < 1 - 1e-12:\n            hi *= 2\n        for _ in range(80):\n            mid = (lo + hi) / 2\n            if mean(values(mid)) < 1:\n                lo = mid\n            else:\n                hi = mid\n        return values((lo + hi) / 2)\n\n    def update(self, losses):\n        if not losses or any(not math.isfinite(v) or v < 0 for v in losses.values()):\n            raise ValueError(\'Probe losses must be nonnegative and finite\')\n        scores = {}\n        for g, value in losses.items():\n            if g not in self.weights:\n                raise ValueError(f\'Unknown probe group: {g}\')\n            if g not in self.initial:\n                self.initial[g] = self.previous[g] = self.smoothed[g] = value\n                scores[g] = 0.\n                continue\n            smooth = self.ema * self.smoothed[g] + (1 - self.ema) * value\n            progress = max(0., (self.smoothed[g] - smooth) / max(self.initial[g], 1e-8))\n            scores[g] = progress if self.mode == \'progress\' else smooth / max(self.initial[g], 1e-8)\n            self.previous[g], self.smoothed[g] = self.smoothed[g], smooth\n        if self.mode in (\'uniform\', \'balanced\'):\n            return self.weights\n        # Groups without probes retain a neutral score; coverage is reported.\n        scores = {g: scores.get(g, 0.) for g in self.groups}\n        peak = max(scores.values())\n        exp = {g: math.exp((v - peak) / self.temperature) for g, v in scores.items()}\n        avg = sum(exp.values()) / len(exp)\n        raw = {g: (1 - self.mix) + self.mix * exp[g] / avg for g in self.groups}\n        self.weights = self._bounded(raw)\n        return self.weights\n\n\ndef write_json(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temp = path.with_suffix(path.suffix + \'.tmp\')\n    temp.write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding=\'utf-8\')\n    temp.replace(path)\n', 'training/train_distill.py': '"""Kaggle pilot: offline-response distillation, with uniform/adaptive group loss.\n\nNo teacher model, generated labels, or benchmark test feedback is used.\nHeavy dependencies are imported only by prepare/train/export commands.\n"""\nimport argparse\nfrom collections import Counter, defaultdict\nfrom importlib.metadata import version\nimport json\nfrom pathlib import Path\nimport random\nimport time\n\nfrom distill_core import (GroupWeights, digest, file_hash, mapped_record, read_records,\n                          split_records, write_json)\n\n\ndef audit(args):\n    fields, groups, examples = Counter(), Counter(), []\n    count = 0\n    rng = random.Random(42)\n    for line, obj in read_records(args.file):\n        count += 1\n        fields.update(obj.keys())\n        groups[str(obj.get(args.group_field, \'<missing>\'))] += 1\n        if len(examples) < args.examples:\n            examples.append({\'line\': line, \'record\': obj})\n        else:\n            index = rng.randrange(count)\n            if index < args.examples:\n                examples[index] = {\'line\': line, \'record\': obj}\n    report = {\'file\': str(Path(args.file).resolve()), \'sha256\': file_hash(args.file),\n              \'records\': count, \'fields\': dict(fields), \'groups\': dict(groups), \'examples\': examples}\n    write_json(args.output, report)\n    print(json.dumps(report, ensure_ascii=False, indent=2))\n\n\ndef prepare(args):\n    if not args.mapping_confirmed:\n        raise ValueError(\'Inspect audit.json first; then explicitly pass --mapping_confirmed with verified field names.\')\n    if args.max_length < 2:\n        raise ValueError(\'max_length must include prompt and response\')\n    out = Path(args.output)\n    if out.exists() and any(out.iterdir()):\n        raise ValueError(\'Prepared output must be empty; choose a new directory to preserve prior manifests.\')\n    from huggingface_hub import HfApi\n    from transformers import AutoTokenizer\n    revision = HfApi().model_info(args.model, revision=args.model_revision).sha\n    tokenizer = AutoTokenizer.from_pretrained(args.model, revision=revision)\n    if tokenizer.eos_token_id is None or not tokenizer.chat_template:\n        raise ValueError(\'Tokenizer requires chat template and EOS\')\n    records, rejected, seen = [], [], set()\n    source_sha = file_hash(args.file)\n    fields = args.prompt_fields.split(\',\')\n    for line, obj in read_records(args.file):\n        record_id = f\'{source_sha}:{line}\'\n        try:\n            r = mapped_record(obj, fields, args.response_field, args.group_field, args.source_field)\n            key = digest([r[\'prompt\'], r[\'response\'], r[\'group\']])\n            if key in seen:\n                raise ValueError(\'duplicate prompt/response/group\')\n            seen.add(key)\n            prefix = tokenizer.apply_chat_template(\n                [{\'role\': \'user\', \'content\': r[\'prompt\']}], tokenize=False,\n                add_generation_prompt=True, enable_thinking=False)\n            prompt_ids = tokenizer.encode(prefix, add_special_tokens=False)\n            response_ids = tokenizer.encode(r[\'response\'], add_special_tokens=False)\n            if not response_ids:\n                raise ValueError(\'empty response tokens\')\n            if response_ids[-1] != tokenizer.eos_token_id:\n                response_ids.append(tokenizer.eos_token_id)\n            if len(prompt_ids) + len(response_ids) > args.max_length:\n                raise ValueError(\'overlength: whole record excluded; no response truncation\')\n            r.update(id=record_id, input_ids=prompt_ids + response_ids,\n                     labels=[-100] * len(prompt_ids) + response_ids,\n                     prompt_tokens=len(prompt_ids), response_tokens=len(response_ids))\n            records.append(r)\n        except ValueError as exc:\n            rejected.append({\'id\': record_id, \'reason\': str(exc)})\n    train, probe, holdout, mapping = split_records(\n        records, args.seed, args.dev_fraction, args.max_train, args.probe_per_group,\n        args.holdout_per_group, args.min_group_train, args.max_groups)\n    out.mkdir(parents=True, exist_ok=True)\n    for name, rows in [(\'train\', train), (\'probe\', probe), (\'holdout\', holdout)]:\n        with (out / f\'{name}.jsonl\').open(\'w\', encoding=\'utf-8\') as stream:\n            for r in rows:\n                stream.write(json.dumps(r, ensure_ascii=False) + \'\\n\')\n    tokenizer.save_pretrained(out / \'tokenizer\')\n    clean_config = {k: v for k, v in vars(args).items() if k != \'func\'}\n    manifest = {\'config\': clean_config, \'model\': args.model, \'model_revision\': revision,\n                \'source_sha256\': source_sha, \'corpus_revision\': args.corpus_revision,\n                \'tokenizer_template_sha256\': digest(tokenizer.chat_template), \'group_mapping\': mapping,\n                \'tokenizer_hashes\': {p.name: file_hash(p) for p in sorted((out / \'tokenizer\').iterdir()) if p.is_file()},\n                \'eligible\': len(records), \'rejected\': len(rejected),\n                \'counts\': {name: len(rows) for name, rows in [(\'train\', train), (\'probe\', probe), (\'holdout\', holdout)]},\n                \'train_groups\': dict(Counter(r[\'train_group\'] for r in train)),\n                \'probe_groups\': dict(Counter(r[\'train_group\'] for r in probe)),\n                \'holdout_groups\': dict(Counter(r[\'train_group\'] for r in holdout)),\n                \'split_hashes\': {name: file_hash(out / f\'{name}.jsonl\') for name in (\'train\', \'probe\', \'holdout\')},\n                \'train_input_tokens\': sum(len(r[\'input_ids\']) for r in train),\n                \'train_response_tokens\': sum(r[\'response_tokens\'] for r in train)}\n    write_json(out / \'rejected.json\', rejected)\n    write_json(out / \'manifest.json\', manifest)\n    write_json(out / \'preview.json\', [{k: r[k] for k in (\'id\', \'prompt\', \'response\', \'group\', \'train_group\')}\n                                     for r in train[:12]])\n    print(json.dumps(manifest, indent=2, ensure_ascii=False))\n    print(\'Inspect preview.json, exclusions and group coverage before training.\')\n\n\ndef per_example_loss(logits, labels):\n    """Response-only mean CE, chunked to avoid fp32 [batch, sequence, vocab]."""\n    import torch\n    import torch.nn.functional as F\n    result = []\n    for row, target in zip(logits, labels):\n        target = target[1:]\n        positions = torch.where(target != -100)[0]\n        if not positions.numel():\n            raise ValueError(\'Sample has no response targets\')\n        total = row.new_zeros((), dtype=torch.float32)\n        for indices in positions.split(128):\n            total = total + F.cross_entropy(row[indices].float(), target[indices], reduction=\'sum\')\n        result.append(total / positions.numel())\n    return torch.stack(result)\n\n\ndef train(args):\n    import os\n    os.environ.setdefault(\'CUBLAS_WORKSPACE_CONFIG\', \':4096:8\')\n    import numpy as np\n    import torch\n    from peft import LoraConfig, PeftModel, get_peft_model, prepare_model_for_kbit_training\n    from transformers import (AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig,\n                              get_cosine_schedule_with_warmup, set_seed)\n    if not torch.cuda.is_available():\n        raise ValueError(\'Train on Kaggle/Colab CUDA GPU; CPU training disabled\')\n    if int(os.environ.get(\'WORLD_SIZE\', \'1\')) != 1:\n        raise ValueError(\'This pilot supports one process/GPU; do not launch with torchrun\')\n    if min(args.steps, args.batch_size, args.grad_accum, args.probe_every, args.save_every, args.log_every, args.rank) < 1:\n        raise ValueError(\'steps, batch, accumulation and intervals must be positive\')\n    if args.learning_rate <= 0 or not 0 <= args.warmup_ratio <= 1:\n        raise ValueError(\'Invalid learning rate/warmup\')\n    prepared, out = Path(args.prepared), Path(args.output)\n    manifest = json.loads((prepared / \'manifest.json\').read_text())\n    for split, expected in manifest[\'split_hashes\'].items():\n        if file_hash(prepared / f\'{split}.jsonl\') != expected:\n            raise ValueError(f\'Prepared split changed: {split}\')\n    for name, expected in manifest[\'tokenizer_hashes\'].items():\n        if file_hash(prepared / \'tokenizer\' / name) != expected:\n            raise ValueError(f\'Prepared tokenizer changed: {name}\')\n    data = {name: [r for _, r in read_records(prepared / f\'{name}.jsonl\')]\n            for name in (\'train\', \'probe\', \'holdout\')}\n    tokenizer = AutoTokenizer.from_pretrained(prepared / \'tokenizer\')\n    tokenizer.pad_token = tokenizer.eos_token\n    set_seed(args.seed)\n    torch.backends.cuda.matmul.allow_tf32 = False\n    torch.backends.cudnn.benchmark = False\n    bf16 = torch.cuda.is_bf16_supported()\n    dtype = torch.bfloat16 if bf16 else torch.float16\n    versions = {p: version(p) for p in (\'torch\', \'transformers\', \'peft\', \'accelerate\', \'numpy\')}\n    if args.four_bit:\n        versions[\'bitsandbytes\'] = version(\'bitsandbytes\')\n    config = {k: v for k, v in vars(args).items() if k not in (\'output\', \'resume\', \'func\')}\n    identity = {\'config\': config, \'manifest_sha256\': file_hash(prepared / \'manifest.json\'),\n                \'source_sha256\': digest([file_hash(Path(__file__)), file_hash(Path(__file__).with_name(\'distill_core.py\'))]),\n                \'versions\': versions, \'gpu\': torch.cuda.get_device_name(0), \'dtype\': str(dtype)}\n    if out.exists() and any(out.iterdir()) and not args.resume:\n        raise ValueError(\'Output is not empty. Use --resume or choose a new run directory.\')\n    checkpoint = None\n    if args.resume:\n        checkpoint = Path(args.resume).resolve()\n        if checkpoint.parent != out.resolve():\n            raise ValueError(\'Resume in the original run output directory\')\n        if json.loads((checkpoint / \'identity.json\').read_text()) != identity:\n            raise ValueError(\'Resume configuration/data/source/environment differs\')\n    out.mkdir(parents=True, exist_ok=True)\n    write_json(out / \'identity.json\', identity)\n    options = {\'revision\': manifest[\'model_revision\'], \'torch_dtype\': dtype,\n               \'device_map\': {\'\': 0}, \'attn_implementation\': \'sdpa\'}\n    if args.four_bit:\n        options[\'quantization_config\'] = BitsAndBytesConfig(\n            load_in_4bit=True, bnb_4bit_quant_type=\'nf4\', bnb_4bit_use_double_quant=True,\n            bnb_4bit_compute_dtype=dtype)\n    model = AutoModelForCausalLM.from_pretrained(manifest[\'model\'], **options)\n    model.config.use_cache = False\n    if args.four_bit:\n        model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)\n    else:\n        model.gradient_checkpointing_enable()\n        model.enable_input_require_grads()\n    if checkpoint:\n        model = PeftModel.from_pretrained(model, checkpoint / \'adapter\', is_trainable=True)\n    else:\n        model = get_peft_model(model, LoraConfig(\n            task_type=\'CAUSAL_LM\', r=args.rank, lora_alpha=args.rank * 2,\n            lora_dropout=0., target_modules=[\'q_proj\', \'k_proj\', \'v_proj\', \'o_proj\']))\n    model.print_trainable_parameters()\n    optimizer = torch.optim.AdamW((p for p in model.parameters() if p.requires_grad),\n                                 lr=args.learning_rate, weight_decay=0.01)\n    scheduler = get_cosine_schedule_with_warmup(optimizer, round(args.steps * args.warmup_ratio), args.steps)\n    scaler = torch.amp.GradScaler(\'cuda\', enabled=not bf16)\n    counts = manifest[\'train_groups\']\n    if args.mode != \'uniform\' and len(counts) < 2:\n        raise ValueError(\'Adaptive/group balancing requires at least two train groups; inspect grouping metadata\')\n    controller = GroupWeights(sorted(counts), counts, args.mode, args.mix,\n                              args.temperature, args.ema, args.weight_floor, args.weight_ceiling)\n    state = {\'step\': 0, \'epoch\': 0, \'offset\': 0, \'train_tokens\': 0, \'response_tokens\': 0,\n             \'probe_tokens\': 0, \'holdout_tokens\': 0, \'elapsed_seconds\': 0., \'probe_seconds\': 0.,\n             \'holdout_seconds\': 0., \'optimizer_updates\': 0, \'peak_vram_bytes\': 0}\n    if checkpoint:\n        # Only load a checkpoint produced by this script from your own run.\n        saved = torch.load(checkpoint / \'state.pt\', map_location=\'cpu\', weights_only=False)\n        optimizer.load_state_dict(saved[\'optimizer\'])\n        scheduler.load_state_dict(saved[\'scheduler\'])\n        scaler.load_state_dict(saved[\'scaler\'])\n        controller.__dict__.update(saved[\'controller\'])\n        state = saved[\'state\']\n        random.setstate(saved[\'python_rng\'])\n        np.random.set_state(saved[\'numpy_rng\'])\n        torch.set_rng_state(saved[\'torch_rng\'])\n        torch.cuda.set_rng_state_all(saved[\'cuda_rng\'])\n        if state[\'step\'] >= args.steps:\n            raise ValueError(\'Checkpoint already reached requested steps\')\n        # Remove stale post-checkpoint log entries after a crash, without duplicating steps.\n        log = out / \'metrics.jsonl\'\n        if log.exists():\n            lines = [json.loads(s) for s in log.read_text().splitlines() if s.strip()]\n            log.write_text(\'\'.join(json.dumps(r) + \'\\n\' for r in lines if r[\'step\'] <= state[\'step\']))\n    start = time.monotonic()\n    prior_elapsed = state[\'elapsed_seconds\']\n    torch.cuda.reset_peak_memory_stats()\n\n    def batch(rows):\n        length = max(len(r[\'input_ids\']) for r in rows)\n        ids = [r[\'input_ids\'] + [tokenizer.pad_token_id] * (length - len(r[\'input_ids\'])) for r in rows]\n        labels = [r[\'labels\'] + [-100] * (length - len(r[\'labels\'])) for r in rows]\n        mask = [[1] * len(r[\'input_ids\']) + [0] * (length - len(r[\'input_ids\'])) for r in rows]\n        return {k: torch.tensor(v, device=\'cuda\') for k, v in\n                [(\'input_ids\', ids), (\'attention_mask\', mask), (\'labels\', labels)]}\n\n    def measure(split):\n        torch.cuda.synchronize()\n        begin = time.monotonic()\n        totals, sizes = defaultdict(float), Counter()\n        model.eval()\n        with torch.inference_mode(), torch.autocast(\'cuda\', dtype=dtype):\n            for r in data[split]:\n                b = batch([r])\n                labels = b.pop(\'labels\')\n                output = model(**b, use_cache=False)\n                loss = per_example_loss(output.logits, labels).item()\n                totals[r[\'train_group\']] += loss\n                sizes[r[\'train_group\']] += 1\n                state[f\'{split}_tokens\'] += len(r[\'input_ids\'])\n                del output\n        model.train()\n        torch.cuda.synchronize()\n        state[f\'{split}_seconds\'] += time.monotonic() - begin\n        return {g: totals[g] / sizes[g] for g in totals}\n\n    def log(value):\n        with (out / \'metrics.jsonl\').open(\'a\') as stream:\n            stream.write(json.dumps({\'step\': state[\'step\'], **value}) + \'\\n\')\n        print(json.dumps({\'step\': state[\'step\'], **value}), flush=True)\n\n    def save():\n        torch.cuda.synchronize()\n        state[\'elapsed_seconds\'] = prior_elapsed + time.monotonic() - start\n        state[\'peak_vram_bytes\'] = max(state[\'peak_vram_bytes\'], torch.cuda.max_memory_allocated())\n        target = out / f"checkpoint-{state[\'step\']}"\n        temp = out / f"checkpoint-{state[\'step\']}.tmp"\n        if target.exists() or temp.exists():\n            raise ValueError(f\'Checkpoint path exists: {target}; preserve or move it before resuming\')\n        temp.mkdir()\n        model.save_pretrained(temp / \'adapter\')\n        tokenizer.save_pretrained(temp / \'adapter\')\n        write_json(temp / \'identity.json\', identity)\n        torch.save({\'optimizer\': optimizer.state_dict(), \'scheduler\': scheduler.state_dict(),\n                    \'scaler\': scaler.state_dict(), \'controller\': controller.__dict__, \'state\': state,\n                    \'python_rng\': random.getstate(), \'numpy_rng\': np.random.get_state(),\n                    \'torch_rng\': torch.get_rng_state(), \'cuda_rng\': torch.cuda.get_rng_state_all()}, temp / \'state.pt\')\n        temp.rename(target)\n        write_json(out / \'latest.json\', {\'checkpoint\': str(target.resolve()), \'state\': state})\n\n    if not checkpoint:\n        initial = measure(\'probe\')\n        controller.update(initial)\n        log({\'event\': \'initial_probe\', \'probe_loss\': initial, \'weights\': controller.weights,\n             \'holdout_loss\': measure(\'holdout\')})\n    model.train()\n    order_epoch, order = None, None\n    while state[\'step\'] < args.steps:\n        optimizer.zero_grad(set_to_none=True)\n        micro_losses = []\n        for _ in range(args.grad_accum):\n            rows = []\n            while len(rows) < args.batch_size:\n                if order_epoch != state[\'epoch\']:\n                    order_epoch = state[\'epoch\']\n                    order = list(range(len(data[\'train\'])))\n                    random.Random(args.seed + state[\'epoch\']).shuffle(order)\n                take = min(args.batch_size - len(rows), len(order) - state[\'offset\'])\n                rows.extend(data[\'train\'][i] for i in order[state[\'offset\']:state[\'offset\'] + take])\n                state[\'offset\'] += take\n                if state[\'offset\'] == len(order):\n                    state[\'epoch\'] += 1\n                    state[\'offset\'] = 0\n            b = batch(rows)\n            labels = b.pop(\'labels\')\n            with torch.autocast(\'cuda\', dtype=dtype):\n                output = model(**b, use_cache=False)\n                losses = per_example_loss(output.logits, labels)\n                weights = losses.new_tensor([controller.weights[r[\'train_group\']] for r in rows])\n                loss = (losses * weights).mean() / args.grad_accum\n            if not torch.isfinite(loss):\n                raise FloatingPointError(\'Non-finite loss; stop and lower LR/check data/precision\')\n            scaler.scale(loss).backward()\n            micro_losses.append(losses.detach().mean().item())\n            state[\'train_tokens\'] += sum(len(r[\'input_ids\']) for r in rows)\n            state[\'response_tokens\'] += sum(r[\'response_tokens\'] for r in rows)\n            del output, losses, loss\n        scaler.unscale_(optimizer)\n        grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.)\n        old_scale = scaler.get_scale()\n        scaler.step(optimizer)\n        scaler.update()\n        if scaler.get_scale() >= old_scale:\n            scheduler.step()\n            state[\'optimizer_updates\'] += 1\n        state[\'step\'] += 1\n        if state[\'step\'] % args.log_every == 0 or state[\'step\'] == args.steps:\n            log({\'event\': \'train\', \'response_ce\': sum(micro_losses) / len(micro_losses),\n                 \'grad_norm\': float(grad_norm), \'learning_rate\': scheduler.get_last_lr()[0],\n                 \'train_tokens\': state[\'train_tokens\'], \'response_tokens\': state[\'response_tokens\'],\n                 \'optimizer_updates\': state[\'optimizer_updates\']})\n        if state[\'step\'] % args.probe_every == 0 or state[\'step\'] == args.steps:\n            losses = measure(\'probe\')\n            if state[\'step\'] >= args.warmup_steps:\n                controller.update(losses)\n            log({\'event\': \'probe\', \'probe_loss\': losses, \'weights\': controller.weights,\n                 \'holdout_loss\': measure(\'holdout\')})\n        if state[\'step\'] % args.save_every == 0 or state[\'step\'] == args.steps:\n            save()\n    model.save_pretrained(out / \'adapter\')\n    tokenizer.save_pretrained(out / \'adapter\')\n    write_json(out / \'summary.json\', {\'mode\': args.mode, \'model\': manifest[\'model\'], \'state\': state,\n                                    \'weights\': controller.weights,\n                                    \'note\': \'Teacher-response CE is not benchmark accuracy/F1. Run SentiBench separately.\'})\n\n\ndef export(args):\n    import torch\n    from peft import PeftModel\n    from transformers import AutoModelForCausalLM, AutoTokenizer\n    run = Path(args.run)\n    identity = json.loads((run / \'identity.json\').read_text())\n    prepared = Path(identity[\'config\'][\'prepared\'])\n    manifest = json.loads((prepared / \'manifest.json\').read_text())\n    target = Path(args.output)\n    if target.exists() and any(target.iterdir()):\n        raise ValueError(\'Export target must be empty\')\n    model = AutoModelForCausalLM.from_pretrained(manifest[\'model\'], revision=manifest[\'model_revision\'],\n                                               torch_dtype=torch.float32, device_map={\'\': \'cpu\'})\n    model = PeftModel.from_pretrained(model, run / \'adapter\').merge_and_unload()\n    model.save_pretrained(target, safe_serialization=True)\n    AutoTokenizer.from_pretrained(run / \'adapter\').save_pretrained(target)\n    write_json(target / \'distillation_run.json\', identity)\n    print(\'Merged model:\', target)\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    commands = parser.add_subparsers(dest=\'command\', required=True)\n    a = commands.add_parser(\'audit\')\n    a.add_argument(\'--file\', required=True)\n    a.add_argument(\'--group_field\', default=\'cluster_id\')\n    a.add_argument(\'--examples\', type=int, default=5)\n    a.add_argument(\'--output\', required=True)\n    a.set_defaults(func=audit)\n    p = commands.add_parser(\'prepare\')\n    p.add_argument(\'--file\', required=True)\n    p.add_argument(\'--output\', required=True)\n    p.add_argument(\'--model\', default=\'Qwen/Qwen3-0.6B\')\n    p.add_argument(\'--model_revision\', default=\'main\')\n    p.add_argument(\'--corpus_revision\', required=True)\n    p.add_argument(\'--prompt_fields\', required=True, help=\'Comma-separated fields in verified order\')\n    p.add_argument(\'--response_field\', required=True)\n    p.add_argument(\'--group_field\', required=True)\n    p.add_argument(\'--source_field\', required=True, help=\'Original input text for leakage-safe grouped splitting\')\n    p.add_argument(\'--mapping_confirmed\', action=\'store_true\')\n    p.add_argument(\'--max_length\', type=int, default=2048)\n    p.add_argument(\'--max_train\', type=int, default=5000, help=\'0 = all eligible training records\')\n    p.add_argument(\'--seed\', type=int, default=42)\n    p.add_argument(\'--dev_fraction\', type=float, default=0.1)\n    p.add_argument(\'--probe_per_group\', type=int, default=4)\n    p.add_argument(\'--holdout_per_group\', type=int, default=4)\n    p.add_argument(\'--min_group_train\', type=int, default=8)\n    p.add_argument(\'--max_groups\', type=int, default=16)\n    p.set_defaults(func=prepare)\n    t = commands.add_parser(\'train\')\n    t.add_argument(\'--prepared\', required=True)\n    t.add_argument(\'--output\', required=True)\n    t.add_argument(\'--mode\', choices=[\'uniform\', \'balanced\', \'loss\', \'progress\'], default=\'uniform\')\n    t.add_argument(\'--steps\', type=int, default=200)\n    t.add_argument(\'--batch_size\', type=int, default=1)\n    t.add_argument(\'--grad_accum\', type=int, default=8)\n    t.add_argument(\'--learning_rate\', type=float, default=1e-4)\n    t.add_argument(\'--rank\', type=int, default=16)\n    t.add_argument(\'--warmup_ratio\', type=float, default=0.05)\n    t.add_argument(\'--warmup_steps\', type=int, default=25, help=\'Before adaptive updates\')\n    t.add_argument(\'--probe_every\', type=int, default=25)\n    t.add_argument(\'--save_every\', type=int, default=100)\n    t.add_argument(\'--log_every\', type=int, default=10)\n    t.add_argument(\'--mix\', type=float, default=0.5)\n    t.add_argument(\'--temperature\', type=float, default=0.05)\n    t.add_argument(\'--ema\', type=float, default=0.5)\n    t.add_argument(\'--weight_floor\', type=float, default=0.5)\n    t.add_argument(\'--weight_ceiling\', type=float, default=2.)\n    t.add_argument(\'--seed\', type=int, default=42)\n    t.add_argument(\'--four_bit\', action=\'store_true\')\n    t.add_argument(\'--resume\', help=\'Own checkpoint directory; restores optimizer/scaler/RNG/controller/cursor\')\n    t.set_defaults(func=train)\n    e = commands.add_parser(\'export\')\n    e.add_argument(\'--run\', required=True)\n    e.add_argument(\'--output\', required=True)\n    e.set_defaults(func=export)\n    args = parser.parse_args()\n    args.func(args)\n\n\nif __name__ == \'__main__\':\n    main()\n', 'requirements-train-kaggle.txt': '# Reuse the platform CUDA PyTorch. Notebook blocks torch/nvidia/triton changes.\n# Known API target; live dependency/import/GPU checks run on Kaggle, not locally.\ntransformers==4.51.3\ntokenizers==0.21.4\nhuggingface-hub==0.30.2\npeft==0.15.2\naccelerate==1.6.0\nsafetensors>=0.5.3,<0.6\n# Optional QLoRA only: bitsandbytes==0.46.1 (not required for ordinary LoRA).\n'}
for relative, source in BUNDLED.items():
    target = BASE / 'source' / relative
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(source, encoding='utf-8')
SCRIPT = BASE / 'source/training/train_distill.py'
REQUIREMENTS = BASE / 'source/requirements-train-kaggle.txt'

def run(arguments, log=None):
    command = [sys.executable, str(SCRIPT), *map(str, arguments)]
    if log is None:
        subprocess.run(command, check=True)
        return
    # Stream output and persist it; failed child raises instead of continuing to next mode.
    with Path(log).open('a', encoding='utf-8') as stream:
        process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                   text=True, bufsize=1)
        try:
            for line in process.stdout:
                print(line, end='', flush=True)
                stream.write(line)
                stream.flush()
            returncode = process.wait()
        except BaseException:
            process.terminate()
            process.wait()
            raise
    if returncode:
        raise subprocess.CalledProcessError(returncode, command)


## Dependency trên Kaggle

Cell cài thư viện train vào runtime hiện tại; không nâng/hạ PyTorch/CUDA.
Pip dry-run dừng nếu kế hoạch thay `torch`, `torchvision`, `torchaudio`, `triton`, hoặc `nvidia-*`.
Mọi code model chạy ở subprocess mới, tránh module cache của notebook sau pip install.
Không cần Hugging Face token cho Qwen public; nếu bị rate-limit, dùng secret `HF_TOKEN`.


In [ ]:
if INSTALL_DEPENDENCIES:
    extra = ['bitsandbytes==0.46.1'] if FOUR_BIT else []
    report = WORK / 'pip-plan.json'
    base = [sys.executable, '-m', 'pip', 'install', '--only-binary=:all:', '-r', str(REQUIREMENTS), *extra]
    subprocess.run([*base, '--dry-run', '--report', str(report)], check=True)
    plan = json.loads(report.read_text())
    names = [p['metadata']['name'].lower().replace('_', '-') for p in plan.get('install', [])]
    forbidden = [n for n in names if n in {'torch', 'torchvision', 'torchaudio', 'triton'} or n.startswith('nvidia-')]
    if forbidden:
        raise RuntimeError(f'Pip would replace CUDA stack: {forbidden}; use a compatible platform runtime.')
    subprocess.run(base, check=True)
else:
    print('Using platform packages; the next runtime cell checks imports.')


In [ ]:
# GPU/dependency validation only; no synthetic model inference.
subprocess.run([sys.executable, '-c', r"""
import json, torch, transformers, peft, accelerate
from importlib.metadata import version
from packaging.version import Version
assert Version(transformers.__version__) >= Version('4.51.0'), 'Qwen3 requires transformers >= 4.51'
assert torch.cuda.is_available(), 'Enable a GPU accelerator'
print(json.dumps({
    'versions': {p: version(p) for p in ('torch', 'transformers', 'peft', 'accelerate', 'huggingface-hub')},
    'gpus': [{'name': torch.cuda.get_device_name(i),
              'vram_gib': torch.cuda.get_device_properties(i).total_memory / 2**30}
             for i in range(torch.cuda.device_count())],
    'bf16': torch.cuda.is_bf16_supported(),
}, indent=2))
"""], check=True)
print('Pilot dùng GPU 0; không tự gộp VRAM của hai T4.')


## Tải đúng file và audit

Dùng `hf_hub_download` cho từng file, tránh lỗi schema khi loader trộn ba teacher.
Revision được khóa; file attach thủ công vẫn được kiểm SHA-256 ở prepare nhưng cần tự xác minh provenance.
Audit lấy mẫu reservoir trải khắp file; không chỉ xem những dòng đầu thường cùng task.


In [ ]:
from huggingface_hub import HfApi, hf_hub_download
if LOCAL_TEACHER_FILE:
    CORPUS = Path(LOCAL_TEACHER_FILE)
    CORPUS_REVISION = 'local-unverified-provenance:' + DATASET_REVISION
else:
    CORPUS_REVISION = HfApi().dataset_info(DATASET, revision=DATASET_REVISION).sha
    CORPUS = Path(hf_hub_download(DATASET, TEACHER_FILE, repo_type='dataset', revision=CORPUS_REVISION))
run(['audit', '--file', CORPUS, '--group_field', 'selected_cluster_id', '--examples', 40,
     '--output', WORK / 'audit.json'], log=WORK / 'audit.log')


## Xác nhận field mapping sau khi đọc audit

Đọc tối thiểu 30–50 mẫu: user text đã nằm trong `instruction` chưa? `input` rỗng có nghĩa gì?
`PROMPT_FIELDS` là thứ tự nối field bằng hai newline; không được nhân đôi user text hoặc demonstration.
Ví dụ **chỉ khi đã xác minh**: `['instruction', 'text']` nếu instruction không chứa câu hỏi cuối;
hoặc `['instruction', 'input']` nếu input là phần cần nối; hoặc `['instruction']` nếu đã chứa đầy đủ.
Giữ nguyên field gốc và đáp án. Không parse/viết lại response teacher.

`SOURCE_FIELD` phải là văn bản nguồn dùng phát hiện cùng input qua nhiều task; không dùng task_name/ID.
`GROUP_FIELD` cần đủ mẫu/nhóm; `cluster_id` có thể rỗng, kiểm tra `selected_cluster_id` trong audit.
Đặt `MAPPING_CONFIRMED=True` sau khi kiểm tra. Đây là bước chặn train sai schema, không tự suy đoán.


In [ ]:
PROMPT_FIELDS = ['instruction', 'text']
RESPONSE_FIELD = 'Qwen/Qwen3-32B_output'
GROUP_FIELD = 'selected_cluster_id'
SOURCE_FIELD = 'text'
MAPPING_CONFIRMED = True

if not MAPPING_CONFIRMED or not PROMPT_FIELDS:
    raise ValueError('Read audit.json, set PROMPT_FIELDS and confirm mapping before prepare.')
PREPARED = WORK / 'prepared'
prepare_args = ['prepare', '--file', CORPUS, '--output', PREPARED, '--model', MODEL,
                '--model_revision', MODEL_REVISION, '--corpus_revision', CORPUS_REVISION,
                '--prompt_fields', ','.join(PROMPT_FIELDS), '--response_field', RESPONSE_FIELD,
                '--group_field', GROUP_FIELD, '--source_field', SOURCE_FIELD, '--mapping_confirmed',
                '--max_length', MAX_LENGTH, '--max_train', MAX_TRAIN, '--seed', SEED,
                '--max_groups', 16, '--probe_per_group', 4, '--holdout_per_group', 4]
request = dict(model=MODEL, model_revision=MODEL_REVISION, corpus_revision=CORPUS_REVISION,
               file=str(CORPUS), prompt_fields=PROMPT_FIELDS, response_field=RESPONSE_FIELD,
               group_field=GROUP_FIELD, source_field=SOURCE_FIELD, max_length=MAX_LENGTH,
               max_train=MAX_TRAIN, seed=SEED)
request_file = WORK / 'prepare-request.json'
if (PREPARED / 'manifest.json').exists():
    if not request_file.exists() or json.loads(request_file.read_text()) != request:
        raise ValueError('Prepare settings differ. Change RUN_TAG instead of reusing old splits.')
    print('Reusing prepared corpus; train checks split hashes.')
else:
    run(prepare_args, log=WORK / 'prepare.log')
    request_file.write_text(json.dumps(request, indent=2))
manifest = json.loads((PREPARED / 'manifest.json').read_text())
print(json.dumps(manifest, ensure_ascii=False, indent=2))
print((PREPARED / 'preview.json').read_text())


## Train baseline và đề xuất

Uniform và progress dùng cùng samples, shuffle, seed, LoRA, steps, response-only mean-per-example loss.
Mọi nhánh đều đo probe/holdout cùng lịch để kiểm soát chi phí đo. Progress cập nhật trọng số bằng EMA
của tiến bộ tương đối trên probe, hỗn hợp với trọng số đều; giới hạn 0,5–2 và chuẩn hóa theo train counts.
Probe được dùng điều phối; holdout tách input độc lập và chỉ ghi log, không cập nhật controller.
Không chọn cấu hình bằng SentiBench-test. Nhóm không có probe vẫn giữ trọng số nền; xem coverage.

Checkpoint gồm adapter, optimizer, scheduler, scaler, RNG, data cursor và controller.
Để resume: restore toàn thư mục WORK vào cùng đường dẫn, giữ source/config/packages và điền RESUME.
Không chỉ restore adapter. Hai nhánh mặc định 200 bước vẫn chỉ là pilot, chưa đi qua toàn train pool.


In [ ]:
for mode in MODES:
    output = WORK / mode
    run_request = dict(model=MODEL, prepared=str(PREPARED), mode=mode, steps=STEPS,
                       batch_size=1, grad_accum=GRAD_ACCUM, learning_rate=LEARNING_RATE,
                       rank=16, seed=SEED, probe_every=PROBE_EVERY,
                       warmup_steps=PROBE_EVERY, save_every=SAVE_EVERY, four_bit=FOUR_BIT)
    request_path = WORK / f'{mode}-request.json'
    if request_path.exists() and json.loads(request_path.read_text()) != run_request:
        raise ValueError('Train settings changed. Choose a new RUN_TAG; do not silently skip an old run.')
    request_path.write_text(json.dumps(run_request, indent=2))
    if (output / 'summary.json').exists():
        print('Completed already:', mode, '— choose a new RUN_TAG to rerun with changed settings.')
        continue
    arguments = ['train', '--prepared', PREPARED, '--output', output, '--mode', mode,
                 '--steps', STEPS, '--batch_size', 1, '--grad_accum', GRAD_ACCUM,
                 '--learning_rate', LEARNING_RATE, '--rank', 16, '--seed', SEED,
                 '--probe_every', PROBE_EVERY, '--warmup_steps', PROBE_EVERY,
                 '--save_every', SAVE_EVERY]
    if FOUR_BIT:
        arguments.append('--four_bit')
    if RESUME.get(mode):
        arguments += ['--resume', RESUME[mode]]
    run(arguments, log=WORK / f'{mode}.log')


## Số liệu pilot

CE là mức khớp đáp án teacher, **không phải Accuracy/Macro-F1/SentiBench score**.
Thời gian gồm train, probe, holdout và các checkpoint trước đó; VRAM ghi peak tensor allocated,
không thay cho tổng VRAM hiển thị bởi nvidia-smi. So token và optimizer_updates giữa các nhánh;
FP16 có thể bỏ optimizer update khi overflow — nếu khác nhiều cần chỉnh LR/precision và chạy lại.
Để tuyên bố cải thiện chất lượng, phải đánh giá checkpoint trên benchmark với evaluator đã kiểm chứng.


In [ ]:
rows = []
for mode in MODES:
    path = WORK / mode / 'summary.json'
    if not path.exists():
        continue
    summary = json.loads(path.read_text())
    state = summary['state']
    events = [json.loads(s) for s in (WORK / mode / 'metrics.jsonl').read_text().splitlines() if s.strip()]
    measured = [r for r in events if r.get('holdout_loss')]
    counts = manifest['holdout_groups']
    def holdout_mean(event):
        values = event['holdout_loss']
        return sum(values[g] * counts[g] for g in values) / sum(counts[g] for g in values)
    rows.append({'mode': mode, 'steps': state['step'], 'updates': state['optimizer_updates'],
                 'initial_holdout_ce': holdout_mean(measured[0]),
                 'final_holdout_ce': holdout_mean(measured[-1]),
                 'train_tokens': state['train_tokens'], 'probe_tokens': state['probe_tokens'],
                 'holdout_tokens': state['holdout_tokens'],
                 'minutes': state['elapsed_seconds'] / 60,
                 'peak_allocated_gib': state['peak_vram_bytes'] / 2**30})
print(json.dumps(rows, indent=2))
(WORK / 'comparison.json').write_text(json.dumps(rows, indent=2))


## Export model và lưu artifacts

Adapter không phải model đầy đủ để truyền trực tiếp cho evaluator hiện tại. Nếu cần, đặt EXPORT_MODE
rồi merge trên CPU; cần RAM cho model float32 và dung lượng đĩa. Export từng nhánh khi dùng.
Sau đó dùng đường dẫn `merged-*` làm `--model_name_or_path` trong pipeline đánh giá đã audit.
Evaluator cũ chưa có chat/non-thinking wrapper và chưa xác nhận giao thức v2; không dùng điểm của nó
để kết luận cải thiện theo paper trước khi kiểm chứng. Notebook này chưa tích hợp SentiBench evaluator v2.

Save Version với outputs trên Kaggle. File ZIP nhỏ gồm log/manifest/split IDs/config/adapter, không gồm
base model, raw corpus, tokenized splits hay optimizer. Resume cần giữ cả WORK bằng Kaggle outputs.
Trước khi xóa session, kiểm tra output thực sự được lưu. Không có đảm bảo session đủ cho full run.


In [ ]:
if EXPORT_MODE:
    if EXPORT_MODE not in MODES:
        raise ValueError('EXPORT_MODE must be a trained mode')
    run(['export', '--run', WORK / EXPORT_MODE, '--output', WORK / f'merged-{EXPORT_MODE}'],
        log=WORK / f'export-{EXPORT_MODE}.log')

# Portable run metadata and adapters; full checkpoints stay in WORK for Save Version.
import zipfile
archive = BASE / f'{RUN_TAG}-{MODEL_TAG}-artifacts.zip'
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as z:
    for path in WORK.rglob('*'):
        if not path.is_file():
            continue
        relative = path.relative_to(WORK)
        if any(p.startswith('checkpoint-') or p.startswith('merged-') for p in relative.parts):
            continue
        if relative.parts[0] == 'prepared' and path.name not in {
            'manifest.json', 'preview.json', 'rejected.json'}:
            continue
        z.write(path, relative)
    for name in ('train', 'probe', 'holdout'):
        ids = []
        with (PREPARED / f'{name}.jsonl').open() as stream:
            for line in stream:
                r = json.loads(line)
                ids.append({k: r[k] for k in ('id', 'source_key', 'group', 'train_group')})
        z.writestr(f'prepared/{name}-ids.json', json.dumps(ids))
    for relative, source in BUNDLED.items():
        z.writestr('source/' + relative, source)
print('Artifacts:', archive)
print('Full resume checkpoints:', WORK)


Nguồn: [corpus v2](https://huggingface.co/datasets/zhang-yice/sentiment-distillation-v2),
[CompEffDist](https://arxiv.org/html/2510.24425v1),
[Qwen3-0.6B](https://huggingface.co/Qwen/Qwen3-0.6B).
Phương pháp progress-weighting là ứng viên nghiên cứu; chưa chứng nhận novelty hay lợi ích.
